In [1]:
import requests
import os
import re
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()
HEADERS = {"User-Agent": os.getenv("SEC_USER_AGENT")}

In [2]:
# Fetch AAPL filing raw HTML  
# Use your working get_filing_text() from before
from src.ingest.ingest_filings import get_cik, get_latest_accession, get_primary_doc_url


In [4]:
cik, company   = get_cik("AAPL")
accession, date, document = get_latest_accession(cik, "10-K")
doc_url        = get_primary_doc_url(cik, accession, "10-K", document)
raw_html       = requests.get(doc_url, headers=HEADERS).text

print(f"Raw HTML size: {len(raw_html):,} chars")

Trying document URL: https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm
Raw HTML size: 1,520,208 chars


In [5]:
# Parse and strip noise the same way ingest does
soup = BeautifulSoup(raw_html, "lxml")
for tag in soup(["script", "style"]): tag.decompose()
for tag in soup.find_all(["ix:header", "header"]): tag.decompose()
for tag in soup.find_all("ix:hidden"): tag.decompose()
for tag in soup.find_all(style=True):
    style = tag.get("style", "").lower().replace(" ", "")
    if "display:none" in style or "visibility:hidden" in style:
        tag.decompose()

full_text = soup.get_text(separator="\n")

/tmp/ipykernel_15277/1040128255.py:2: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(raw_html, "lxml")


In [6]:
# ── Diagnostic 1: Search for Item markers in the raw text ─────────────────
print("\n=== All lines containing 'Item' or 'ITEM' ===")
lines = full_text.splitlines()
item_lines = [
    (i, line.strip()) 
    for i, line in enumerate(lines) 
    if re.search(r'\bITEM\b|\bItem\b', line.strip()) 
    and len(line.strip()) < 120  # headers are short
]
print(f"Found {len(item_lines)} candidate header lines\n")
for lineno, line in item_lines[:30]:  # show first 30
    print(f"  Line {lineno:5d}: {repr(line)}")


=== All lines containing 'Item' or 'ITEM' ===
Found 47 candidate header lines

  Line   136: 'Item 1.'
  Line   139: 'Item 1A.'
  Line   142: 'Item 1B.'
  Line   145: 'Item 1C.'
  Line   148: 'Item 2.'
  Line   151: 'Item 3.'
  Line   154: 'Item 4.'
  Line   158: 'Item 5.'
  Line   161: 'Item 6.'
  Line   164: 'Item 7.'
  Line   167: 'Item 7A.'
  Line   170: 'Item 8.'
  Line   173: 'Item 9.'
  Line   176: 'Item 9A.'
  Line   179: 'Item 9B.'
  Line   182: 'Item 9C.'
  Line   186: 'Item 10.'
  Line   189: 'Item 11.'
  Line   192: 'Item 12'
  Line   196: 'Item 13'
  Line   200: 'Item 14.'
  Line   204: 'Item 15.'
  Line   207: 'Item 16.'
  Line   213: 'Item 1.\xa0\xa0\xa0\xa0Business'
  Line   343: 'Item 1A.\xa0\xa0\xa0\xa0Risk Factors'
  Line   468: 'Item 1B.\xa0\xa0\xa0\xa0Unresolved Staff Comments'
  Line   470: 'Item 1C.\xa0\xa0\xa0\xa0Cybersecurity'
  Line   491: 'Item 2.\xa0\xa0\xa0\xa0Properties'
  Line   494: 'Item 3.\xa0\xa0\xa0\xa0Legal Proceedings'
  Line   505: 'Item 4.\xa0\x